In [ ]:
 #주로 사용하는 library import
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy import stats

#실행6_2_1


In [ ]:
#정규성 검정 (자료의 수가 작을 때 : n<30 , 자료의 수가 크면 생략)
#가설) H0:데이터가 정규분포를 따른다 / H1:데이터가 정규분포를 따르지 않는다
#data
data_1samp = [23, 21, 19, 22, 20, 18, 24, 25, 21, 20]  #리스트, 넘파이의 어레이, 판다스의 시리즈(판다스의 변수 한 줄) 형식 다 가능함

#Shapiro-Wilk 정규성 검정
statistic, p_value = stats.shapiro(data_1samp)  #ShapiroResult라는 클래스에 statistic과 pvalue 값이 들어있음

#결과 출력
print(f"Shapiro-Wilk's W: {statistic:.3f}")
print(f"P-value: {p_value}")

#해석
if p_value < 0.05:
 print("귀무가설을 기각합니다. 데이터는 정규분포를 따르지 않습니다.")
else:
 print("귀무가설을 채택합니다. 데이터는 정규분포를 따릅니다.")

Shapiro-Wilk's W: 0.973
P-value: 0.9146187003727988
귀무가설을 채택합니다. 데이터는 정규분포를 따릅니다.


In [ ]:
stats.shapiro(data_1samp)

ShapiroResult(statistic=np.float64(0.9726917845035304), pvalue=np.float64(0.9146187003727988))

#실행6_2_2

In [ ]:
#일표본t검정
#가설) H0: 모평균이 mu0와 같다 / H1: 모평균이 mu0와 다르다
#mu0
mu = 20

#일표본t-검정 수행
t_statistic, p_value = stats.ttest_1samp(data_1samp, mu)  #클래스로 출력
      #1차원 수치형 데이터(리스트, NumPy 배열, Pandas Series) 입력
      #alternative 옵션 'two-sided', 'less', 'greater' 설정 가능. 기본은 양측

#신뢰구간 계산
confidence = 0.95
n = len(data_1samp)
df = n - 1
mean = np.mean(data_1samp)
std_err = stats.sem(data_1samp) #표준오차
t_crit = stats.t.ppf((1 + confidence)/2,df) #임계값(양축)  #ppf: 누적분포 함수의 역함수

margin = t_crit * std_err
ci_lower = mean - margin
ci_upper = mean + margin

#결과 출력
print(f"Sample mean: {mean:.3f}")
print(f"T-statistic: {t_statistic:.3f}")
print(f"P-value: {p_value:.4f}")
print(f"95% CI: ({ci_lower:.3f}, {ci_upper:.3f}")

#해석
if p_value < 0.05:
 print(f"귀무가설을 기각합니다. 평균이 {mu}과(와) 유의하게 다릅니다.")
else:
 print(f"귀무가설을 채택합니다. 평균이 {mu}과(와) 차이가 없습니다.")

Sample mean: 21.300
T-statistic: 1.857
P-value: 0.0963
95% CI: (19.716, 22.884
귀무가설을 채택합니다. 평균이 20과(와) 차이가 없습니다.


#실행6_3_1

In [ ]:
#등분산검정 (검정 결과에 따라 옵션 설정)
#가설) H0:두 집단이 등분산이다 / H1:두 집단 등분산이 아니다
#data
group1 = [23, 21, 19, 22, 20, 18, 24, 25, 21, 20]
group2 = [30, 28, 32, 29, 31, 33, 34, 35, 32, 30]

#Levene 등분산 검정
statistic, p_value = stats.levene(group1, group2)

#결과 출력
print(f"Levene's W: {statistic:.3f}")
print(f"P-value: {p_value:.4f}")

#해석
if p_value < 0.05:
 print("귀무가설을 기각합니다. 두 집단은 등분산이 아닙니다(이분산).")
else:
 print("귀무가설을 채택합니다. 두 집단은 등분산입니다.")

Levene's W: 0.032
P-value: 0.8602
귀무가설을 채택합니다. 두 집단은 등분산입니다.


#실행6_3_2

In [ ]:
#등분산 여부
equal_var=True

#독립표본 t검정
#equal_var 옵션 True = Student's t-test, False = Welch's t-test
t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=equal_var)

#기본 통계 계산
mean1, mean2 = np.mean(group1), np.mean(group2)
n1, n2 = len(group1), len(group2)
var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1) #ddof=1 표본분산

#신뢰구간 계산
confidence = 0.95
diff = mean1 - mean2

if equal_var: #equal_var == True일 때 실행
 pooled_var = ((n1 - 1)*var1 + (n2 - 1)*var2) / (n1 + n2 - 2)
 se = np.sqrt(pooled_var * (1/n1 + 1/n2))
 df = n1 + n2 - 2
else: #equal_var == False일 때 실행
 se = np.sqrt(var1/n1 + var2/n2)
 df = (var1/n1 + var2/n2)**2/((var1**2)/((n1**2)*(n1-1))+(var2**2)/((n2**2)*(n2-1)))  #(시험에서) 외울 필요없음
 t_crit = stats.t.ppf((1 + confidence) / 2, df)
ci_lower = diff - t_crit * se
ci_upper = diff + t_crit * se

#결과 출력
print(f"Group1 mean: {mean1:.2f}")
print(f"Group2 mean: {mean2:.2f}")
print(f"Mean difference (Group1 - Group2): {diff:.2f}")
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.4f}")
print(f"95% CI for mean difference: ({ci_lower:.3f}, {ci_upper:.3f})")

#해석
if p_value < 0.05:
 print("귀무가설을 기각합니다. 두 그룹의 평균 차이는 유의미합니다.")
else:
 print("귀무가설을 채택합니다. 두 그룹의 평균 차이는 유의미하지 않습니다.")

Group1 mean: 21.30
Group2 mean: 31.40
Mean difference (Group1 - Group2): -10.10
T-statistic: -10.185
P-value: 0.0000
95% CI for mean difference: (-12.343, -7.857)
귀무가설을 기각합니다. 두 그룹의 평균 차이는 유의미합니다.


#실행6_4_1

In [ ]:
#data
before = [70, 65, 80, 90, 78, 84, 73, 72, 68, 75]  #순서가 대응되도록 입력해야 함
after = [65, 60, 75, 85, 70, 80, 70, 68, 66, 72]

#대응표본 t-검정
t_statistic, p_value = stats.ttest_rel(before, after)

#차이 계산
diff = np.array(before) - np.array(after)
mean_diff = np.mean(diff)
std_err = stats.sem(diff) # 표준오차
df = len(diff) - 1

#신뢰구간 계산 (95%)
confidence = 0.95
t_crit = stats.t.ppf((1 + confidence) / 2, df)
margin = t_crit * std_err
ci_lower = mean_diff - margin
ci_upper = mean_diff + margin

#결과 출력
print(f"Mean (before - after): {mean_diff:.3f}")
print(f"T-statistic: {t_statistic:.3f}")
print(f"P-value: {p_value:.4f}")
print(f"95% CI: ({ci_lower:.3f}, {ci_upper:.3f})")

#p-value가 0.05보다 작으면 평균 차이가 유의미하다고 판단
if p_value < 0.05:
 print("귀무가설을 기각합니다. 전후 데이터에 유의미한 차이가 있습니다.")
else:
 print("귀무가설을 채택합니다. 전후 데이터에 차이가 없습니다.")

Mean (before - after): 4.400
T-statistic: 8.450
P-value: 0.0000
95% CI: (3.222, 5.578)
귀무가설을 기각합니다. 전후 데이터에 유의미한 차이가 있습니다.


In [ ]:
print(std_err)

0.5206833117271102


#연습6_1_1

In [ ]:
#data : raw data 형식 (students 데이터)
from google.colab import drive
#구글 드라이브 마운트(연결)
drive.mount('/content/drive')
#드라이브에 저장된 파일 경로 지정
file_path = '/content/drive/MyDrive/Colab_ex_data/students.csv'
#데이터프레임 만들기
df_students = pd.read_csv(file_path)

#일표본t검정
#가설) H0: 모평균이 mu0와 같다 / H1: 모평균이 mu0와 다르다
mu = 60
df_math=df_students['math'] #데이터프레임의 시리즈 형식

#일표본t-검정 수행
t_statistic, p_value = stats.ttest_1samp(df_math, mu)  #클래스로 출력
      #1차원 수치형 데이터(리스트, NumPy 배열, Pandas Series) 입력
      #alternative 옵션 'two-sided', 'less', 'greater' 설정 가능. 기본은 양측

#신뢰구간 계산
confidence = 0.95
n = len(df_math)
df = n - 1
mean = np.mean(df_math)
std_err = stats.sem(df_math) #표준오차
t_crit = stats.t.ppf((1 + confidence)/2,df) #임계값(양축)  #ppf: 누적분포 함수의 역함수

margin = t_crit * std_err
ci_lower = mean - margin
ci_upper = mean + margin

#결과 출력
print(f"Sample mean: {mean:.3f}")
print(f"T-statistic: {t_statistic:.3f}")
print(f"P-value: {p_value:.4f}")
print(f"95% CI: ({ci_lower:.3f}, {ci_upper:.3f})")

#해석
if p_value < 0.05:
 print(f"귀무가설을 기각합니다. 평균이 {mu}과(와) 유의하게 다릅니다.")
else:
 print(f"귀무가설을 채택합니다. 평균이 {mu}과(와) 차이가 없습니다.")

Mounted at /content/drive
Sample mean: 59.820
T-statistic: -0.063
P-value: 0.9500
95% CI: (54.086, 65.554)
귀무가설을 채택합니다. 평균이 60과(와) 차이가 없습니다.


In [ ]:
n

50

#연습6_1_2

In [ ]:
group1 = df_students[df_students['major']=='A']['math']
group2 = df_students[df_students['major']=='B']['math']

#Levene 등분산 검정
statistic, p_value = stats.levene(group1, group2)
#결과 출력
print(f"Levene's W: {statistic:.3f}")
print(f"P-value: {p_value:.4f}")
#해석
if p_value < 0.05:
 print("귀무가설을 기각합니다. 두 집단은 등분산이 아닙니다(이분산).")
else:
 print("귀무가설을 채택합니다. 두 집단은 등분산입니다.")

Levene's W: 0.576
P-value: 0.4531
귀무가설을 채택합니다. 두 집단은 등분산입니다.


In [ ]:
#등분산 여부
equal_var=True

#독립표본 t검정
#equal_var 옵션 True = Student's t-test, False = Welch's t-test
t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=equal_var)

#기본 통계 계산
mean1, mean2 = np.mean(group1), np.mean(group2)
n1, n2 = len(group1), len(group2)
var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1) #ddof=1 표본분산

#신뢰구간 계산
confidence = 0.95
diff = mean1 - mean2

if equal_var: #equal_var == True일 때 실행
 pooled_var = ((n1 - 1)*var1 + (n2 - 1)*var2) / (n1 + n2 - 2)
 se = np.sqrt(pooled_var * (1/n1 + 1/n2))
 df = n1 + n2 - 2
else: #equal_var == False일 때 실행
 se = np.sqrt(var1/n1 + var2/n2)
 df = (var1/n1 + var2/n2)**2/((var1**2)/((n1**2)*(n1-1))+(var2**2)/((n2**2)*(n2-1)))  #(시험에서) 외울 필요없음

t_crit = stats.t.ppf((1 + confidence) / 2, df)
ci_lower = diff - t_crit * se
ci_upper = diff + t_crit * se

#결과 출력
print(f"Group1 mean: {mean1:.2f}")
print(f"Group2 mean: {mean2:.2f}")
print(f"Mean difference (Group1 - Group2): {diff:.2f}")
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.4f}")
print(f"95% CI for mean difference: ({ci_lower:.3f}, {ci_upper:.3f})")

#해석
if p_value < 0.05:
 print("귀무가설을 기각합니다. 두 그룹의 평균 차이는 유의미합니다.")
else:
 print("귀무가설을 채택합니다. 두 그룹의 평균 차이는 유의미하지 않습니다.")

Group1 mean: 56.35
Group2 mean: 57.77
Mean difference (Group1 - Group2): -1.42
T-statistic: -0.201
P-value: 0.8420
95% CI for mean difference: (-15.802, 12.959)
귀무가설을 채택합니다. 두 그룹의 평균 차이는 유의미하지 않습니다.


#연습6_1_3

In [ ]:
#data
before = [51, 61, 46, 31, 26, 52, 62, 47, 32, 27, 66, 81, 90, 79, 52, 51, 81, 91, 21, 51, 66,
46, 52, 82, 92, 22, 52, 67, 47, 62, 47, 32, 82, 92, 45, 47, 74, 57, 64, 46, 48, 75,
58, 65, 79, 88, 77, 80, 89, 78]
after = df_math

#대응표본 t-검정
t_statistic, p_value = stats.ttest_rel(before, after)

#차이 계산
diff = np.array(before) - np.array(after)
mean_diff = np.mean(diff)
std_err = stats.sem(diff) # 표준오차
df = len(diff) - 1

#신뢰구간 계산 (95%)
confidence = 0.95
t_crit = stats.t.ppf((1 + confidence) / 2, df)
margin = t_crit * std_err
ci_lower = mean_diff - margin
ci_upper = mean_diff + margin

#결과 출력
print(f"Mean (before - after): {mean_diff:.3f}")
print(f"T-statistic: {t_statistic:.3f}")
print(f"P-value: {p_value:.4f}")
print(f"95% CI: ({ci_lower:.3f}, {ci_upper:.3f})")

#p-value가 0.05보다 작으면 평균 차이가 유의미하다고 판단
if p_value < 0.05:
 print("귀무가설을 기각합니다. 전후 데이터에 유의미한 차이가 있습니다.")
else:
 print("귀무가설을 채택합니다. 전후 데이터에 차이가 없습니다.")

Mean (before - after): 0.360
T-statistic: 2.701
P-value: 0.0095
95% CI: (0.092, 0.628)
귀무가설을 기각합니다. 전후 데이터에 유의미한 차이가 있습니다.


In [ ]:
#참고(시험에는 안나옴)
#대응표본 t검정 시 데이터의 수가 작은 경우 정규성 검정 진행
diff = np.array(before) - np.array(after)
statistic, p_value = stats.shapiro(diff)
print(statistic, p_value)

0.5879476394143679 1.295382084228124e-10
